# 01 — Data Extraction
**Project:** NYC Property Rolling Sales — 4 Tableau Dashboards Pipeline

This notebook is **Stage 1 of 5**.

| Stage | Notebook | Goal |
|---|---|---|
| 1 | `extraction.ipynb` | Load raw CSV, audit shape, types, sample, save raw snapshot |
| 2 | `cleaning.ipynb` | Fix dtypes, handle missing, drop/impute, derive columns |
| 3 | `eda.ipynb` | Univariate / Bivariate / Multivariate visual exploration |
| 4 | `statistical.ipynb` | Descriptive + inferential stats (ANOVA, chi-sq, correlation) |
| 5 | `final_load_prep.ipynb` | Final Tableau-ready CSV(s) for 4 dashboards |

### Dataset
[NYC DOF Rolling Sales — Kaggle](https://www.kaggle.com/datasets/new-york-city/nyc-property-sales).
Each row = one property sold in NYC over a 12-month rolling window.

### Source file expected
`../data/nyc-rolling-sales.csv` (place the unzipped CSV in a `data/` folder one level above the notebooks).

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print('pandas :', pd.__version__)
print('numpy  :', np.__version__)

## 2. File paths

In [ ]:
# Adjust if your folder layout differs
RAW_PATH    = Path('../data/nyc-rolling-sales.csv')
OUT_DIR     = Path('../data')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Raw file exists :', RAW_PATH.exists())
print('Size (MB)       :', round(RAW_PATH.stat().st_size / 1024 / 1024, 2) if RAW_PATH.exists() else 'N/A')

## 3. Load the raw CSV

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
print('Shape:', df_raw.shape)
df_raw.head()

## 4. Schema audit (column names & dtypes)

In [ ]:
schema = pd.DataFrame({
    'column'  : df_raw.columns,
    'dtype'   : df_raw.dtypes.astype(str).values,
    'n_unique': [df_raw[c].nunique(dropna=True) for c in df_raw.columns],
    'n_null'  : df_raw.isnull().sum().values,
    'sample'  : [df_raw[c].dropna().astype(str).head(2).tolist() for c in df_raw.columns],
})
schema

### Observations
* `Unnamed: 0` is a leftover index column → drop in cleaning.
* Numeric columns `SALE PRICE`, `LAND SQUARE FEET`, `GROSS SQUARE FEET` arrive as **strings** because of `' - '` placeholders.
* `SALE DATE` is a string and must be parsed as datetime.
* `BOROUGH` is coded 1–5 (1=Manhattan, 2=Bronx, 3=Brooklyn, 4=Queens, 5=Staten Island).
* `EASE-MENT` and `APARTMENT NUMBER` are mostly blank — candidates for drop.

## 5. Quick stats on the raw frame

In [ ]:
print('Rows         :', len(df_raw))
print('Columns      :', df_raw.shape[1])
print('Memory (MB)  :', round(df_raw.memory_usage(deep=True).sum()/1024/1024, 2))
print('Duplicates   :', df_raw.duplicated().sum())

## 6. Borough decode (so analysts reading the raw file are not confused)

In [ ]:
BOROUGH_MAP = {1: 'Manhattan', 2: 'Bronx', 3: 'Brooklyn', 4: 'Queens', 5: 'Staten Island'}
df_raw['BOROUGH_NAME_PREVIEW'] = df_raw['BOROUGH'].map(BOROUGH_MAP)
df_raw['BOROUGH_NAME_PREVIEW'].value_counts()

## 7. Save a raw snapshot for reproducibility

In [ ]:
# We persist exactly what we read so the cleaning notebook can start from a frozen file.
RAW_SNAPSHOT = OUT_DIR / 'nyc_sales_raw.parquet'
df_raw.drop(columns=['BOROUGH_NAME_PREVIEW']).to_parquet(RAW_SNAPSHOT, index=False)
print('Saved:', RAW_SNAPSHOT, '|', round(RAW_SNAPSHOT.stat().st_size/1024/1024, 2), 'MB')

## ✅ Stage 1 done
Next → open **`cleaning.ipynb`**.